In [1]:
import numpy as np
import pandas as pd

In [3]:
df = pd.read_csv('tmdb_5000_movies.csv'); df.head(1)

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2009-12-10,2787965087,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800


In [4]:
new_df = df[['original_title', 'genres', 'overview']]

In [5]:
new_df.head(1)

,original_title,genres,overview
0,Avatar,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","In the 22nd century, a paraplegic Marine is di..."


In [7]:
import ast

df_clean = new_df.copy()

def extract_primary_genre(genres_str):
    try: 
        genres_list = ast.literal_eval(genres_str)
        if len(genres_list) > 0:
            return genres_list[0]['name']
    except (ValueError, SyntaxError):
        pass
    return None

df_clean['genre'] = df_clean['genres'].apply(extract_primary_genre)

In [8]:
df_clean = df_clean.dropna(subset=['genre', 'overview'])
df_clean = df_clean[df_clean['overview'].str.strip() != '']

df_clean = df_clean.reset_index(drop=True)

In [9]:
print(f"Rows before cleaning: {len(new_df)}")
print(f"Rows after cleaning: {len(df_clean)}")
print(f"Number of unique genres: {df_clean['genre'].nunique()}")
df_clean[['original_title', 'genre', 'overview']].head(5)

Rows before cleaning: 4803
Rows after cleaning: 4771
Number of unique genres: 20


,original_title,genre,overview
0,Avatar,Action,"In the 22nd century, a paraplegic Marine is di..."
1,Pirates of the Caribbean: At World's End,Adventure,"Captain Barbossa, long believed to be dead, ha..."
2,Spectre,Action,A cryptic message from Bond’s past sends him o...
3,The Dark Knight Rises,Action,Following the death of District Attorney Harve...
4,John Carter,Action,"John Carter is a war-weary, former military ca..."


In [13]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.preprocessing import LabelEncoder


vocab_size = 10000
max_len = 100


tokens = Tokenizer(num_words=vocab_size, oov_token="<oov>")
tokens.fit_on_texts(df_clean['overview'])

seq = tokens.texts_to_sequences(df_clean['overview'])
X = pad_sequences(seq, maxlen=max_len, padding='post', truncating='post')

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(df_clean['genre'])

In [14]:
print("Vocabulary size used:", min(vocab_size, len(tokens.word_index) + 1))
print("Shape of X (movies x max words):", X.shape)
print("Shape of y (movies,):", y.shape)
print("Example original overview:", df_clean['overview'].iloc[0])
print("Example tokenized+padded:", X[0])
print("Example genre label:", df_clean['genre'].iloc[0], "->", y[0])

Vocabulary size used: 10000
Shape of X (movies x max words): (4771, 100)
Shape of y (movies,): (4771,)
Example original overview: In the 22nd century, a paraplegic Marine is dispatched to the moon Pandora on a unique mission, but becomes torn between following orders and protecting an alien civilization.
Example tokenized+padded: [   7    2 5495  312    3    1 1796    9 3001    4    2 1681    1   15
    3 1160  155   23   82 1071  113  537 1609    5 2177   13  387 2745
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0]
Example genre label: Action -> 0


In [16]:
from tensorflow.keras.layers import Embedding, Attention, Dense, Input, GlobalMaxPooling1D
from tensorflow.keras.models import Model 

num_classes = len(label_encoder.classes_)

i = Input(shape=(max_len, ), name='overview_input')
E = Embedding(input_dim=vocab_size, output_dim=64, name='word_embedding')(i)
A = Attention(name='attention')([E, E])
pooled = GlobalMaxPooling1D(name='meaning_vector')(A)
D = Dense(num_classes, activation='softmax')(pooled)

model = Model(inputs=i, outputs=D)
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ overview_input      │ (None, 100)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ word_embedding      │ (None, 100, 64)   │    640,000 │ overview_input[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attention           │ (None, 100, 64)   │          0 │ word_embedding[0… │
│ (Attention)         │                   │            │ word_embedding[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ meaning_vector      │ (None, 64)        │          0 │ attention[0][0]   │
│ (GlobalMaxPooling1… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 20)        │      1,300 │ meaning_vector[0… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 641,300 (2.45 MB)

 Trainable params: 641,300 (2.45 MB)

 Non-trainable params: 0 (0.00 B)

In [17]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [20]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history = model.fit(
    X_train, y_train,
    validation_data = (X_test, y_test),
    epochs=100,
    batch_size=32
)

Epoch 1/100
120/120 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - accuracy: 0.6693 - loss: 1.2575 - val_accuracy: 0.3550 - val_loss: 2.4621
Epoch 2/100
120/120 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.6879 - loss: 1.1692 - val_accuracy: 0.3696 - val_loss: 2.4987
Epoch 3/100
120/120 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.7146 - loss: 1.0844 - val_accuracy: 0.3508 - val_loss: 2.5651
Epoch 4/100
120/120 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.7335 - loss: 1.0043 - val_accuracy: 0.3508 - val_loss: 2.6530
Epoch 5/100
120/120 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.7571 - loss: 0.9236 - val_accuracy: 0.3382 - val_loss: 2.7030
Epoch 6/100
120/120 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.7945 - loss: 0.8437 - val_accuracy: 0.3246 - val_loss: 2.8094
Epoch 7/100
120/120 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.8132 - loss: 0.7689 - val_accuracy: 0.3236 - val_loss: 2.8789
Epoch 8/100
120/120 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.8436 - loss: 0.6965 - val_acc

KeyboardInterrupt: 

In [19]:
from tensorflow.keras.models import Model

# A second model that outputs the 64-dim vector instead of the genre prediction
vector_model = Model(inputs=model.input, outputs=model.get_layer('meaning_vector').output)

# Get the meaning vector for every movie in our cleaned dataset
movie_vectors = vector_model.predict(X, batch_size=64)

print("Shape of all movie vectors:", movie_vectors.shape)
print("Example vector (first 10 numbers of Avatar):", movie_vectors[0][:10])

75/75 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step
Shape of all movie vectors: (4771, 64)
Example vector (first 10 numbers of Avatar): [-0.04886547  1.2007422   0.65807474 -0.08251245  0.30300716  0.48266345
  0.83988696 -0.17551257  0.03221205  0.08513918]


In [21]:
from sklearn.metrics.pairwise import cosine_similarity

# Precompute similarity between every pair of movies (4771 x 4771 matrix)
similarity_matrix = cosine_similarity(movie_vectors)

def recommend(title, top_n=5):
    # Find the index of the movie matching the title (case-insensitive)
    matches = df_clean[df_clean['original_title'].str.lower() == title.lower()]
    if matches.empty:
        print(f"'{title}' not found in dataset.")
        return
    
    idx = matches.index[0]
    
    # Get similarity scores for this movie vs all others
    scores = list(enumerate(similarity_matrix[idx]))
    
    # Sort by similarity, skip itself (highest score), take top_n
    scores = sorted(scores, key=lambda x: x[1], reverse=True)
    scores = [s for s in scores if s[0] != idx][:top_n]
    
    print(f"Movies similar to '{df_clean['original_title'].iloc[idx]}' ({df_clean['genre'].iloc[idx]}):\n")
    for i, score in scores:
        print(f"  {df_clean['original_title'].iloc[i]}  —  similarity: {score:.2f}  ({df_clean['genre'].iloc[i]})")

# Test it
recommend("Avatar")

Movies similar to 'Avatar' (Action):

  Captain America: The First Avenger  —  similarity: 1.00  (Action)
  American Ninja 2: The Confrontation  —  similarity: 1.00  (Action)
  Scott Pilgrim vs. the World  —  similarity: 0.99  (Action)
  Congo  —  similarity: 0.99  (Action)
  Ghostbusters  —  similarity: 0.99  (Action)


In [22]:
import numpy as np

norms = np.linalg.norm(movie_vectors, axis=1)
print("Min norm:", norms.min(), "Max norm:", norms.max(), "Mean norm:", norms.mean())

# Check similarity spread across ALL movie pairs (not just Avatar)
sample = similarity_matrix[np.triu_indices(len(similarity_matrix), k=1)]
print("Similarity scores — min:", sample.min(), "max:", sample.max(), "mean:", sample.mean())

Min norm: 3.0067215 Max norm: 9.118594 Mean norm: 4.0908556
Similarity scores — min: -0.25488108 max: 0.9999998 mean: 0.7567497


In [24]:
# Rebuild a fresh model (same architecture) so we start from scratch
i = Input(shape=(max_len,), name='overview_input')
E = Embedding(input_dim=vocab_size, output_dim=64, name='word_embedding')(i)
A = Attention(name='attention')([E, E])
pooled = GlobalMaxPooling1D(name='meaning_vector')(A)
D = Dense(num_classes, activation='softmax')(pooled)

model = Model(inputs=i, outputs=D)
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Stop early - epoch 4 was the sweet spot before val_loss started rising
history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=4,
    batch_size=32
)

# Re-extract vectors from this less-overfit model
vector_model = Model(inputs=model.input, outputs=model.get_layer('meaning_vector').output)
movie_vectors = vector_model.predict(X, batch_size=64)
similarity_matrix = cosine_similarity(movie_vectors)

# Re-check spread
sample = similarity_matrix[np.triu_indices(len(similarity_matrix), k=1)]
print("Similarity scores — min:", sample.min(), "max:", sample.max(), "mean:", sample.mean())

recommend("Avatar")

Epoch 1/4
120/120 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - accuracy: 0.2492 - loss: 2.3584 - val_accuracy: 0.2524 - val_loss: 2.2498
Epoch 2/4
120/120 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.2607 - loss: 2.2152 - val_accuracy: 0.2649 - val_loss: 2.2140
Epoch 3/4
120/120 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.3111 - loss: 2.1522 - val_accuracy: 0.2880 - val_loss: 2.1798
Epoch 4/4
120/120 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.3577 - loss: 2.0477 - val_accuracy: 0.3079 - val_loss: 2.2012
75/75 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step
Similarity scores — min: 0.27446836 max: 0.99999815 mean: 0.86540556
Movies similar to 'Avatar' (Action):

  The Dark Knight  —  similarity: 1.00  (Drama)
  Sweet Sweetback's Baadasssss Song  —  similarity: 1.00  (Action)
  Sunshine  —  similarity: 1.00  (Science Fiction)
  The Fast and the Furious: Tokyo Drift  —  similarity: 1.00  (Action)
  Brigham City  —  similarity: 1.00  (Crime)


In [25]:
from tensorflow.keras.layers import GlobalAveragePooling1D, Dropout
from tensorflow.keras.callbacks import EarlyStopping

i = Input(shape=(max_len,), name='overview_input')
E = Embedding(input_dim=vocab_size, output_dim=64, name='word_embedding')(i)
A = Attention(name='attention')([E, E])
pooled = GlobalAveragePooling1D(name='meaning_vector')(A)
pooled_drop = Dropout(0.3)(pooled)
D = Dense(num_classes, activation='softmax')(pooled_drop)

model = Model(inputs=i, outputs=D)
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

early_stop = EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)

history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=15,
    batch_size=32,
    callbacks=[early_stop]
)

vector_model = Model(inputs=model.input, outputs=model.get_layer('meaning_vector').output)
movie_vectors = vector_model.predict(X, batch_size=64)
similarity_matrix = cosine_similarity(movie_vectors)

sample = similarity_matrix[np.triu_indices(len(similarity_matrix), k=1)]
print("Similarity scores — min:", sample.min(), "max:", sample.max(), "mean:", sample.mean())

recommend("Avatar")

Epoch 1/15
120/120 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - accuracy: 0.2369 - loss: 2.4127 - val_accuracy: 0.2524 - val_loss: 2.2757
Epoch 2/15
120/120 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.2416 - loss: 2.2825 - val_accuracy: 0.2524 - val_loss: 2.2403
Epoch 3/15
120/120 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.2440 - loss: 2.2471 - val_accuracy: 0.2639 - val_loss: 2.2274
Epoch 4/15
120/120 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.2531 - loss: 2.2336 - val_accuracy: 0.2586 - val_loss: 2.2177
Epoch 5/15
120/120 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.2752 - loss: 2.2145 - val_accuracy: 0.2869 - val_loss: 2.2029
Epoch 6/15
120/120 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.2757 - loss: 2.1877 - val_accuracy: 0.2733 - val_loss: 2.1899
Epoch 7/15
120/120 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.2835 - loss: 2.1632 - val_accuracy: 0.2775 - val_loss: 2.1747
Epoch 8/15
120/120 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.3077 - loss: 2.1293 - val_accuracy: 0

In [27]:
i = Input(shape=(max_len,), name='overview_input')
E = Embedding(input_dim=vocab_size, output_dim=32, name='word_embedding')(i)  # 64 -> 32 dims
A = Attention(name='attention')([E, E])
pooled = GlobalAveragePooling1D(name='meaning_vector')(A)
pooled_drop = Dropout(0.4)(pooled)  # 0.3 -> 0.4
D = Dense(num_classes, activation='softmax')(pooled_drop)

model = Model(inputs=i, outputs=D)
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

early_stop = EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)

history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=15,
    batch_size=32,
    callbacks=[early_stop]
)

vector_model = Model(inputs=model.input, outputs=model.get_layer('meaning_vector').output)
movie_vectors = vector_model.predict(X, batch_size=64)
similarity_matrix = cosine_similarity(movie_vectors)

sample = similarity_matrix[np.triu_indices(len(similarity_matrix), k=1)]
print("Similarity scores — min:", sample.min(), "max:", sample.max(), "mean:", sample.mean())

recommend("Avatar")
recommend("The Dark Knight Rises")

Epoch 1/15
120/120 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - accuracy: 0.2330 - loss: 2.5350 - val_accuracy: 0.2524 - val_loss: 2.3077
Epoch 2/15
120/120 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.2400 - loss: 2.3253 - val_accuracy: 0.2429 - val_loss: 2.2548
Epoch 3/15
120/120 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.2521 - loss: 2.2793 - val_accuracy: 0.2524 - val_loss: 2.2366
Epoch 4/15
120/120 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.2385 - loss: 2.2643 - val_accuracy: 0.2639 - val_loss: 2.2255
Epoch 5/15
120/120 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.2455 - loss: 2.2504 - val_accuracy: 0.2796 - val_loss: 2.2140
Epoch 6/15
120/120 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.2526 - loss: 2.2358 - val_accuracy: 0.2869 - val_loss: 2.2059
Epoch 7/15
120/120 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.2613 - loss: 2.2127 - val_accuracy: 0.2681 - val_loss: 2.1973
Epoch 8/15
120/120 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.2733 - loss: 2.2024 - val_accuracy: 0.

In [29]:
i = Input(shape=(max_len,), name='overview_input')
E = Embedding(input_dim=vocab_size, output_dim=32, name='word_embedding', mask_zero=True)(i)
A = Attention(name='attention')([E, E])
pooled = GlobalAveragePooling1D(name='meaning_vector')(A)
pooled_drop = Dropout(0.4)(pooled)
D = Dense(num_classes, activation='softmax')(pooled_drop)

model = Model(inputs=i, outputs=D)
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

early_stop = EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)

history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=15,
    batch_size=32,
    callbacks=[early_stop]
)

vector_model = Model(inputs=model.input, outputs=model.get_layer('meaning_vector').output)
movie_vectors = vector_model.predict(X, batch_size=64)
similarity_matrix = cosine_similarity(movie_vectors)

sample = similarity_matrix[np.triu_indices(len(similarity_matrix), k=1)]
print("Similarity scores — min:", sample.min(), "max:", sample.max(), "mean:", sample.mean())

recommend("Avatar")
recommend("The Dark Knight Rises")

Epoch 1/15
120/120 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step - accuracy: 0.2503 - loss: 2.7533 - val_accuracy: 0.2702 - val_loss: 2.3319
Epoch 2/15
120/120 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.2552 - loss: 2.2858 - val_accuracy: 0.2524 - val_loss: 2.2465
Epoch 3/15
120/120 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.2510 - loss: 2.2527 - val_accuracy: 0.2545 - val_loss: 2.2321
Epoch 4/15
120/120 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.2765 - loss: 2.2269 - val_accuracy: 0.2545 - val_loss: 2.2208
Epoch 5/15
120/120 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.2780 - loss: 2.2118 - val_accuracy: 0.2555 - val_loss: 2.2091
Epoch 6/15
120/120 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.2859 - loss: 2.1935 - val_accuracy: 0.2848 - val_loss: 2.1952
Epoch 7/15
120/120 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.2945 - loss: 2.1703 - val_accuracy: 0.2953 - val_loss: 2.1817
Epoch 8/15
120/120 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.3223 - loss: 2.1316 - val_accuracy: 0.

In [31]:
import pickle

# Save the trained model
model.save('movie_model.keras')

# Save the tokenizer (needed to process new text later, if you ever add it)
with open('tokenizer.pkl', 'wb') as f:
    pickle.dump(tokens, f)

# Save the precomputed movie vectors + similarity matrix + movie metadata
with open('movie_data.pkl', 'wb') as f:
    pickle.dump({
        'movie_vectors': movie_vectors,
        'similarity_matrix': similarity_matrix,
        'titles': df_clean['original_title'].values,
        'genres': df_clean['genre'].values
    }, f)

print("Saved: movie_model.keras, tokenizer.pkl, movie_data.pkl")

Saved: movie_model.keras, tokenizer.pkl, movie_data.pkl
